<div align="center">

### RR Skillverse — Free Learning Handbook
**by Raushan Ranjan**

*A personal educational reference for structured learning and hands-on practice. Shared for learning purposes only — not a commercial product or paid service.*

</div>

---

# Module 1 — Data Science + Financial Data Analysis

**AI & Machine Learning: Advanced Engineering with Cybersecurity**

*Module 1 of 12 · This notebook starts the RR Finance system that every later module extends*

## What we are actually building, across all 12 modules

This is **not** a sequence of 12 unrelated demos. It is **one financial AI system, built incrementally**, the way a real engineering team would build it. Every module adds a new capability to the *same* running system — never a fresh, disconnected toy example. By Module 12 (Final Capstone), everything below is wired together into one integrated, secured, deployed application.

| # | Module | What it adds to the ONE system we are building |
|---|---|---|
| **1** | **Data Science + Financial Data Analysis** *(this notebook)* | **Explore and validate our RR Finance dataset, decide whether we trust it, build the first baseline model, and add the first security control.** |
| 2 | Deep Neural Networks | A stronger tabular model (ANN) benchmarked against this module's baseline, plus a new image-processing capability (CNN), plus the first inference-time attack (adversarial examples) |
| 3 | NLP + Financial Text AI | Reading unstructured financial text (news, filings) into the system, with PII protection |
| 4 | Generative AI, LLMs & RLHF | A local LLM the system can reason with, fine-tuned and red-teamed |
| 5 | RAG, LangChain & AI Agents | Retrieval and autonomous tool-use layered on top of the LLM |
| 6 | Explainable & Responsible AI | Auditability for every model built in Modules 1–5 |
| 7 | Federated & Privacy-Preserving ML | Training across multiple institutions without centralising raw data |
| 8 | Multimodal AI | Speech and document understanding added to the same pipeline |
| 9 | Graph Neural Networks | Fraud-ring and network-level reasoning across the whole customer graph |
| 10 | MLOps & On-Prem Deployment | Everything above, served as real, monitored, secured APIs |
| 11 | AI Cybersecurity | A formal threat model and security audit of the entire system built so far |
| 12 | **Final Capstone** | **The one integrated, deployed, secured production system** |

**The consequence for how you should use this notebook:** every artifact you produce here — the validated dataset, the trained baseline model, the anomaly screen — is **saved to disk at the end of this notebook** and is **loaded, not recreated, by Module 2 onward**. Run this notebook first, all the way through, before opening Module 2.

## What Module 1 is actually FOR

Right now we have one CSV and no trust in it yet. The entire point of this notebook is to answer one question honestly: ***can we trust this data enough to build on it?*** We inspect it, question it, find its problems, decide what to do about them, and only then hand off a dataset (and a documented list of its limitations) to Module 2. "Trusted" does not mean "perfect" — it means *we know exactly what is wrong with it and have made a deliberate decision about each issue.*

## How every lesson is taught

Six questions, asked of every technique, every time:

1. **What problem are we solving?**
2. **Why does it matter in finance?**
3. **Why this technique — what alternatives exist?**
4. **What do the numbers/parameters actually mean?**
5. **What is happening mathematically?**
6. **What happens if we change it?**

Read the markdown, run the code cell immediately below it, inspect the printed output/plot, connect it back to the question that motivated it, *then* move on. Nothing here is a disconnected snippet — each cell exists because the previous cell's output raised the question it answers.


## Setup — run this cell first (it is a REAL, runnable cell, not just instructions)

**What was wrong before:** the setup instructions were shown as a *markdown* code block — text that looks like a code cell but cannot be executed, because Markdown cells never run. Clicking "Run" on it did nothing, and `Run All` then hit `ModuleNotFoundError` on the very next cell because nothing had actually been installed.

**The fix below is an actual code cell** using Jupyter's `%pip` magic command, which installs packages straight into the kernel this notebook is running on (this is the officially recommended way to install packages from inside a notebook — safer than a plain `!pip install`, which can silently target the wrong Python environment).

**Is it safe to leave in `Run All` every time?** Yes. `pip` is idempotent: if a package is already installed at the right version, it prints `Requirement already satisfied` and finishes in a second or two — it will not reinstall, break, or slow down a normal `Run All` in any noticeable way. You do **not** need to comment this cell out or remember to skip it.

**If you prefer to set up the environment once, from a terminal, instead:**
```bash
python -m venv .venv
# Windows: .venv\Scripts\activate
# macOS/Linux: source .venv/bin/activate
pip install numpy pandas matplotlib seaborn scikit-learn imbalanced-learn umap-learn joblib jupyter
```
Both approaches end up in the same place — pick whichever fits how you teach.


In [ ]:
%pip install -q numpy pandas matplotlib seaborn scikit-learn imbalanced-learn umap-learn joblib
print("Setup complete -- if you saw 'Requirement already satisfied' lines above, that is expected and fine.")


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import json
import joblib

SEED = 42
np.random.seed(SEED)
rng = np.random.default_rng(SEED)

# Create the shared project folders this whole 12-module course will keep using
Path("data").mkdir(exist_ok=True)
Path("artifacts").mkdir(exist_ok=True)

print("numpy:", np.__version__, "| pandas:", pd.__version__)
print("Project folders ready: data/, artifacts/")


---
## Foundation first: why Module 1 starts here

Before regression, classification or Isolation Forest, we need to understand the problem we are actually trying to solve — this is the map for the entire 12-module system.

> **Business problem:** *"A financial organisation wants to make better risk decisions from data — and it needs the resulting AI system to be trustworthy and secure. What do we build first?"*

```
Business problem → Evidence/data → Financial meaning → Features → Algorithm → Python → Evaluation → Security controls
```

### Analogy 1 — Building a house
You do not start a house by choosing the paint. You decide what the building must do, inspect the ground, design the structure, choose materials. AI engineering follows the same sequence: **problem → data → representation → algorithm → implementation → validation → security.**

### Analogy 2 — A doctor
A doctor starts with the patient's complaint, observations, measurements, diagnosis — not with a sophisticated treatment chosen because it sounds advanced. The business problem is the complaint; data is the observation; features are the measurable signals; the model is part of the diagnostic machinery.

### Analogy 3 — A detective
A detective does not guess the culprit and then search for evidence. Evidence comes first: inspect it, check whether it is trustworthy, find patterns, test a hypothesis, *then* predict. **This notebook is that detective work, applied to our own dataset.**

### Analogy 4 — A toolbox
An algorithm is not a trophy. A hammer is not "better" than a screwdriver — the right tool depends on the task. Regression answers *how much*; classification answers *which class*; clustering asks *which observations resemble each other*; anomaly detection asks *which observations look unusual*.

### The algorithm decision tree

| Question | Technique | Financial example | Why it fits |
|---|---|---|---|
| How much? | Regression | Expected loan amount | Continuous numeric output |
| Which class / likelihood? | Classification | Default vs. no-default | Known target label |
| What groups exist? | K-Means / DBSCAN | Customer behaviour segments | No target label required |
| Can we simplify many dimensions? | PCA / UMAP | Explore high-dimensional patterns | Compress for analysis/visualisation |
| What looks unusual? | Isolation Forest | Potentially suspicious records | Works without a fraud label for every record |
| What if training evidence is manipulated? | Poisoning experiment | Label flipping / corrupted training data | Tests the security of the learning process |

### Why we deliberately do not start with deep learning

A neural network can learn complex patterns, but complexity does not compensate for a weak foundation. If the feature definition is wrong, the label is wrong, the test set leaks information, or the training data has unaddressed quality problems, a more powerful model (Module 2 onward) simply makes the wrong result more sophisticated. **Module 1 is the foundation layer everything else in this course stands on — which is exactly why the rest of this notebook interrogates our actual dataset instead of assuming it is fine.**


---
## Loading our real RR Finance dataset

| Question | Answer |
|---|---|
| **1. What problem are we solving?** | We have one real CSV of loan applicants. Before any modelling, we need to know exactly what is in it. |
| **2. Why does it matter in finance?** | Every downstream decision — which model to train, which threshold to set, which control to add — is only as good as the data underneath it. |
| **3. Why start with structure, not statistics?** | You cannot interpret a mean or a correlation until you know the shape of the table: how many rows, how many columns, what type each column is. |
| **4. What do the numbers mean?** | Row count tells us how much evidence we have. Column dtypes tell us whether a "numeric-looking" column is actually stored as text (a common real-world data bug). |
| **5. What is happening mathematically?** | Nothing yet — this is inspection, not computation. Resist the urge to model before you have looked. |
| **6. What happens if we change it?** | Not applicable yet — this cell only reads and describes what already exists. |

**Place the dataset at `data/rr_finance_module1_dataset.csv` in your project folder before running the cell below** (the file that came with this handbook).


In [ ]:
DATA_PATH = Path("data/rr_finance_module1_dataset.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Could not find {DATA_PATH.resolve()}.\n"
        "Place rr_finance_module1_dataset.csv inside a 'data' folder next to this notebook, then re-run this cell."
    )

df = pd.read_csv(DATA_PATH)
print("Loaded:", DATA_PATH.resolve())
print("Shape:", df.shape)
df.head(10)


In [ ]:
print("Column types:")
print(df.dtypes)


### Column reference — every field has a business meaning

| Column | Meaning | Why it may matter |
|---|---|---|
| `customer_id` | Unique customer identifier | Not a model input -- identifier only |
| `age` | Customer age in years | Life-stage / stability proxy |
| `annual_income` | Customer annual income | Repayment capacity |
| `monthly_debt` | Existing monthly debt obligations | Debt burden |
| `loan_amount` | Requested loan amount | Exposure size |
| `loan_term_months` | Requested repayment period | Duration of exposure |
| `credit_score` | Creditworthiness indicator | Historical credit-risk signal |
| `employment_years` | Years employed | Income-stability proxy |
| `account_age_months` | Relationship length | Customer history/context |
| `num_previous_loans` | Previous borrowing count | Credit-behaviour context |
| `previous_defaults` | Historical default count | Strong risk signal |
| `debt_to_income` | monthly debt ÷ monthly income | Normalises debt by earning capacity |
| `loan_to_income` | loan amount ÷ annual income | Normalises requested exposure |
| `default` | Target: 1 = default, 0 = no default | What our classifiers learn to predict |

Only **800 rows.** That is small for a production model, and we say so plainly rather than pretending otherwise — the rest of this notebook treats that as a known limitation to document, not hide. It is enough to teach every technique in this module correctly; it is not enough to claim production-grade statistical confidence. If this system moves toward real deployment, growing this dataset is one of the first priorities — see the note on that at the end of this notebook.


---
## Lesson 1 — Data quality: missing values, duplicates, ranges

| Question | Answer |
|---|---|
| **1. What problem are we solving?** | Find the structural problems -- missing data, duplicate records, impossible values -- before they silently corrupt a model. |
| **2. Why does it matter in finance?** | A model trained on duplicated customers or impossible values can produce misleading, costly decisions, and you may not find out until it is already deployed. |
| **3. Why these three checks?** | Missingness, duplication and range sanity are the fastest, highest-value checks you can run on any new dataset -- they catch the most common real-world data problems in minutes. |
| **4. What do the numbers mean?** | Zero missing values and zero duplicates is a *good* sign, but it is not proof of correctness -- a column can be 100% populated and still be wrong, which is why Lesson 2 goes further. |
| **5. What is happening mathematically?** | Straightforward counting and range comparison; no modelling yet. |
| **6. What happens if we change it?** | If we found missing values here, we would need a documented imputation or exclusion strategy before proceeding -- skipping that step silently teaches the model whatever pattern the missingness happens to correlate with. |


In [ ]:
print("Missing values per column:")
print(df.isna().sum())
print("\nDuplicate rows:", df.duplicated().sum())
print("Duplicate customer_ids:", df["customer_id"].duplicated().sum())


In [ ]:
df.describe().round(2)


**First trust checkpoint:** zero missing values, zero duplicate rows, zero duplicate customer IDs. That is a genuinely good sign for a real dataset -- but "clean" is not the same as "correct." The next lesson checks whether the *values* make business sense, not just whether they exist.


---
## Lesson 2 — Logical consistency: does the data make business sense?

| Question | Answer |
|---|---|
| **1. What problem are we solving?** | A value can be present, numeric, and in a plausible *individual* range, and still be wrong when checked *against another field*. |
| **2. Why does it matter in finance?** | An underwriting model trained on logically inconsistent records learns whatever pattern the inconsistency happens to create -- a silent, hard-to-detect source of bias. |
| **3. Why this technique?** | Cross-field consistency checks: does `employment_years` fit inside a plausible working life given `age`? Do our engineered ratios (`debt_to_income`, `loan_to_income`) actually match what the raw fields imply? |
| **4. What do the numbers mean?** | We use `age - 16` as a generous plausibility ceiling for years of employment (allowing for an unusually early working start) -- this is a business assumption we are stating explicitly, not a universal law. |
| **5. What is happening mathematically?** | Simple row-wise comparisons between columns; no modelling. |
| **6. What happens if we change it?** | A stricter ceiling (e.g. `age - 18`) would flag more rows; a looser one would flag fewer -- the threshold itself is a judgement call worth documenting, exactly like every other parameter in this course. |


In [ ]:
# Cross-field sanity check 1: can this many years of employment fit inside this person's life?
implausible_employment = df[df["employment_years"] > (df["age"] - 16)]

print(f"Rows where employment_years exceeds a plausible working life for their age: "
      f"{len(implausible_employment)} of {len(df)} ({len(implausible_employment)/len(df):.2%})")
implausible_employment[["customer_id", "age", "employment_years"]].head(10)


In [ ]:
# Cross-field sanity check 2: do the engineered ratio columns actually match the raw fields they claim to derive from?
recomputed_dti = df["monthly_debt"] / (df["annual_income"] / 12)
recomputed_lti = df["loan_amount"] / df["annual_income"]

dti_mismatch = (recomputed_dti - df["debt_to_income"]).abs() > 0.01
lti_mismatch = (recomputed_lti - df["loan_to_income"]).abs() > 0.01

print("debt_to_income mismatches vs. recomputed value:", dti_mismatch.sum())
print("loan_to_income mismatches vs. recomputed value:", lti_mismatch.sum())
print("\nBoth engineered columns check out -- they were derived correctly from the raw fields.")
print("This is the SAME discipline as Module 1's Feature Engineering lesson: never trust a derived")
print("column just because it looks reasonable -- recompute it and compare.")


### Decision: what do we do with the 34 flagged rows?

We found **34 rows (4.25%)** where `employment_years` does not plausibly fit the customer's `age`. This is a real, documented data-quality issue -- exactly the kind of thing Module 1 exists to surface. Three honest options, in order of how we'd actually handle this in a real project:

1. **Investigate the source** -- was this a data-entry error, a different definition of "employment years" (e.g. total career vs. current job), or a genuine outlier? We don't have that context here, so we can't resolve it definitively.
2. **Quarantine, don't silently drop** -- flag these rows for human review rather than deleting them outright (deleting real customer records without investigation is its own risk).
3. **Proceed, but document the caveat** -- for this teaching notebook, we keep all 800 rows for the modelling lessons below (removing 4.25% of an already-small 800-row dataset has real cost), but we tag them so any later analysis can filter them out or treat them separately.

This is the same "layered defence, not silent trust" principle Lesson 8 (Poisoning Defence) below will generalise into a full architecture.


In [ ]:
df["flag_employment_age_inconsistent"] = (df["employment_years"] > (df["age"] - 16))
print("Flag added. Rows flagged:", df["flag_employment_age_inconsistent"].sum())


---
## Lesson 3 — Exploratory analysis: distributions, target balance, outliers

| Question | Answer |
|---|---|
| **1. What problem are we solving?** | Understand the shape of every important field, and how skewed our target class is, before choosing modelling techniques. |
| **2. Why does it matter in finance?** | The degree of class imbalance directly determines which techniques (Lesson 6) are even necessary, and outliers can silently dominate a model that is not robust to them. |
| **3. Why this technique?** | Histograms for shape, a target bar chart for balance, and an IQR-based scan for outliers are the standard first pass on any new numeric dataset. |
| **4. What do the numbers mean?** | The IQR (interquartile range) method flags a value as a possible outlier if it falls more than 1.5x the IQR beyond the 25th/75th percentile -- a common convention, not a law. |
| **5. What is happening mathematically?** | Nothing beyond descriptive statistics and a standard outlier heuristic. |
| **6. What happens if we change it?** | A stricter IQR multiplier (e.g. 3.0 instead of 1.5) flags fewer, more extreme outliers -- worth adjusting based on how conservative your review process needs to be. |


In [ ]:
print("Target distribution:")
print(df["default"].value_counts())
print(df["default"].value_counts(normalize=True).round(4))

fig, axes = plt.subplots(1, 3, figsize=(13, 3.3))
axes[0].hist(df["debt_to_income"], bins=30, color="tab:blue")
axes[0].set_title("Debt-to-income distribution")
axes[1].hist(df["credit_score"], bins=30, color="tab:orange")
axes[1].set_title("Credit score distribution")
axes[2].bar(["No default", "Default"], df["default"].value_counts().sort_index(), color=["tab:green", "tab:red"])
axes[2].set_title(f"Target class balance ({df['default'].mean():.1%} default)")
plt.tight_layout(); plt.show()


**Read this honestly:** our default rate is **35.1%** -- this is a *moderately* imbalanced problem, not the "6% rare event" scenario some textbook examples use. That changes how urgently we need imbalance-handling techniques in Lesson 6 (still worth demonstrating, but the accuracy trap is less dramatic here than it would be for a true rare-event problem like fraud detection). Say what the data actually shows, not what a tutorial assumes it should show.


In [ ]:
# IQR-based outlier scan on the key numeric fields
outlier_summary = {}
for col in ["annual_income", "monthly_debt", "loan_amount", "credit_score", "age"]:
    q1, q3 = df[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n_outliers = ((df[col] < lower) | (df[col] > upper)).sum()
    outlier_summary[col] = n_outliers
    print(f"{col:16s} range [{df[col].min():>10.0f}, {df[col].max():>10.0f}]  IQR outliers: {n_outliers}")

print("\nA handful of outliers in monthly_debt and credit_score is expected in real financial data --")
print("high earners with unusually large debt, or a small number of exceptionally strong/weak credit")
print("histories. Small counts like these do not by themselves indicate a data-quality problem.")


---
## Lesson 4 — Does the label itself make business sense?

| Question | Answer |
|---|---|
| **1. What problem are we solving?** | Before trusting `default` as a target to predict, check whether it behaves the way real default risk *should* behave against known risk factors. |
| **2. Why does it matter in finance?** | If a customer's history of previous defaults does not correlate with a *higher* default rate here, that is a serious red flag about label quality -- not something a fancier model can fix. |
| **3. Why this technique?** | A `groupby` on a known risk factor, checked against the target rate, is a fast, interpretable sanity test any domain expert can validate at a glance. |
| **4. What do the numbers mean?** | We expect: more previous defaults -> higher default rate; higher credit score -> lower default rate. These are basic, well-established directional relationships in credit risk. |
| **5. What is happening mathematically?** | Conditional means -- `P(default=1 \| group)` for each group -- nothing more sophisticated than that, deliberately, because the whole point is a check any non-technical stakeholder can also read. |
| **6. What happens if we change it?** | If either relationship ran backwards, we would have to stop and investigate the label-generation process before trusting this dataset for *any* modelling -- this check comes before, not after, model-building. |


In [ ]:
print("Default rate by number of previous defaults:")
print(df.groupby("previous_defaults")["default"].agg(["mean", "count"]))

df["credit_score_bucket"] = pd.cut(df["credit_score"], bins=[400, 600, 650, 700, 750, 900])
print("\nDefault rate by credit score bucket:")
print(df.groupby("credit_score_bucket", observed=True)["default"].agg(["mean", "count"]))


**Both relationships run in the expected direction:** default rate rises with prior defaults (31.6% → 50.0% → 53.8%) and falls as credit score improves (51.3% → 24.8% across buckets). This is the single most important check in this notebook -- it means the target column is behaving the way real credit risk behaves, even though the dataset is small and our eventual model accuracy will be modest. **A label that fails this check is not worth modelling, no matter how clean the rest of the table looks.**

### Second trust checkpoint

| Check | Result | Verdict |
|---|---|---|
| Missing values | 0 | Pass |
| Duplicate rows / IDs | 0 / 0 | Pass |
| Engineered ratios match raw fields | Yes | Pass |
| Employment-vs-age consistency | 34 rows (4.25%) flagged | Documented, not hidden |
| Label direction vs. previous_defaults | Correct direction | Pass |
| Label direction vs. credit_score | Correct direction | Pass |

**Decision: this dataset is trusted enough to proceed**, with two documented caveats carried forward explicitly: (1) it is small (800 rows), so treat every metric below as directional evidence, not a production-grade guarantee; (2) 4.25% of rows have a flagged employment/age inconsistency, tagged but not removed. This is what "trusted" means in real engineering -- not flawless, but *known*.


In [ ]:
VALIDATED_PATH = Path("data/rr_finance_module1_dataset.csv")   # same canonical path every later module reads
df.to_csv(VALIDATED_PATH, index=False)
print("Saved validated dataset (with quality flags added) to:", VALIDATED_PATH.resolve())


---
## Lesson 5 — Feature engineering: translating financial reasoning into model inputs

| Question | Answer |
|---|---|
| **1. What problem are we solving?** | Raw fields often do not express the relationship we actually care about. |
| **2. Why does it matter in finance?** | ₹30,000 monthly debt means something different to a customer earning ₹40,000 versus ₹400,000 -- relative measures can be more informative than raw amounts. |
| **3. Why this technique?** | Domain-informed transformations (ratios, counts, rates) -- the right transformation depends on the business problem. |
| **4. What do the numbers mean?** | `debt_to_income = 0.30` means monthly debt consumes 30% of monthly income. |
| **5. What is happening mathematically?** | A ratio changes scale from an absolute quantity to a relative quantity, making customers more comparable. |
| **6. What happens if we change it?** | A badly defined feature can introduce leakage or a misleading signal -- never use information that would only be available *after* the event you are predicting. |

We already verified in Lesson 2 that `debt_to_income` and `loan_to_income` are correctly derived. Here we check *which* engineered and raw features actually correlate with our now-validated target -- the first genuinely quantitative signal check in this notebook.


In [ ]:
numeric_cols = [
    "age", "annual_income", "monthly_debt", "loan_amount", "loan_term_months", "credit_score",
    "employment_years", "account_age_months", "num_previous_loans", "previous_defaults",
    "debt_to_income", "loan_to_income", "default",
]
correlations = df[numeric_cols].corr(numeric_only=True)["default"].drop("default").sort_values()
print("Correlation of each feature with the default target:")
print(correlations.round(3))

plt.figure(figsize=(7, 4))
correlations.plot(kind="barh", color=["tab:red" if v < 0 else "tab:blue" for v in correlations])
plt.title("Which features actually correlate with default?")
plt.xlabel("Correlation with default")
plt.tight_layout(); plt.show()


**Read this honestly too:** the strongest single correlations (`loan_to_income`, `debt_to_income`, `previous_defaults`) sit around **0.15-0.20** -- real, directionally correct, but modest. This tells us in advance not to expect a spectacular single-number "AUC" from a linear model in Lesson 7 -- and it will not be spectacular, which is a realistic, honest outcome for a small, moderately-signalled financial dataset, not a failure of the modelling.


---
## Lesson 6 — Regression: predicting a continuous financial quantity

| Question | Answer |
|---|---|
| **1. What problem are we solving?** | Predict a numeric quantity -- here, the requested `loan_amount` -- from a customer's income/debt/credit profile. |
| **2. Why does it matter in finance?** | Financial planning and risk management often need *amounts*, not only yes/no decisions. |
| **3. Why this technique?** | Ridge and Lasso are linear-regression variants with regularisation -- fast, interpretable baselines. |
| **4. What do the parameters mean?** | `alpha` controls regularisation strength. It is a hyperparameter to **validate**, which is exactly what `RidgeCV` does below rather than asserting a fixed value. |
| **5. What is happening mathematically?** | OLS minimises squared prediction error; Ridge adds an L2 penalty on coefficient size; Lasso adds an L1 penalty (and can zero out coefficients entirely). |
| **6. What happens if we change it?** | Larger regularisation shrinks coefficients more; the useful value is data-dependent, which is why we validate it instead of guessing. |

**Fix carried over from the previous version of this lesson:** `root_mean_squared_error` is used directly -- the older pattern of passing `squared=False` to `mean_squared_error` raises a `TypeError` on scikit-learn >= 1.4, where that parameter was removed.


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Ridge, Lasso, RidgeCV
from sklearn.metrics import root_mean_squared_error, r2_score

X_reg = df[["annual_income", "monthly_debt", "credit_score", "employment_years"]]
y_reg = df["loan_amount"]

Xr_train, Xr_test, yr_train, yr_test = train_test_split(X_reg, y_reg, test_size=0.2, random_state=SEED)

ridge = Ridge(alpha=1.0)
ridge.fit(Xr_train, yr_train)
ridge_pred = ridge.predict(Xr_test)

print("Ridge (alpha=1.0)")
print("  R^2:  ", round(r2_score(yr_test, ridge_pred), 4))
print("  RMSE: ", round(root_mean_squared_error(yr_test, ridge_pred), 2))


In [ ]:
ridge_cv = RidgeCV(alphas=[0.001, 0.01, 0.1, 1, 10, 100, 1000], cv=5)
ridge_cv.fit(Xr_train, yr_train)
ridge_cv_pred = ridge_cv.predict(Xr_test)

print("RidgeCV selected alpha via 5-fold cross-validation:", ridge_cv.alpha_)
print("  R^2:  ", round(r2_score(yr_test, ridge_cv_pred), 4))
print("  RMSE: ", round(root_mean_squared_error(yr_test, ridge_cv_pred), 2))


**Honest read:** R² around 0.20 means these four features explain roughly a fifth of the variation in requested loan amount -- modest, and a realistic finding, not a bug. It tells us requested loan amount is driven by more than just income/debt/credit/tenure (perhaps the specific purpose of the loan, or the applicant's own preference), which is useful business knowledge in itself: it tells a real underwriting team *not* to over-rely on a simple regression here without more context features.


In [ ]:
lasso = Lasso(alpha=0.1)
lasso.fit(Xr_train, yr_train)
print("Ridge coefficients:", ridge.coef_.round(2))
print("Lasso coefficients:", lasso.coef_.round(2))


> **Data governance flag, found during this exploration (not assumed going in):** `age` is present in this dataset and correlates with several other fields, but age is a **protected attribute** under fair-lending regulation in most jurisdictions (e.g. the U.S. Equal Credit Opportunity Act). Using it directly as a model input creates real regulatory and fairness exposure, even in a synthetic-data teaching exercise -- the habit matters more than this specific dataset does. **Decision: `age` stays in `df` for exploration and later fairness auditing (Module 6 will use it to test the model for age-based disparate impact), but it is deliberately excluded from `FEATURES` below.** This is exactly the kind of finding Module 1's "explore first, trust second" process exists to surface.

---
## Lesson 7 — Classification: predicting default (the model every later module benchmarks against)

| Question | Answer |
|---|---|
| **1. What problem are we solving?** | Predict `default` (1) vs. no default (0) -- **this is the classifier Module 2's ANN will be benchmarked against.** |
| **2. Why does it matter in finance?** | Financial decisions often have asymmetric consequences: missing a true default is usually costlier than investigating an extra false alert. |
| **3. Why this technique?** | Logistic Regression is a fast, explainable baseline -- exactly what you want *before* reaching for a neural network. |
| **4. What do the parameters mean?** | The classification threshold (default 0.5) is a **decision policy**, not a law -- it should reflect the business cost of false positives vs. false negatives, which we tune explicitly below. |
| **5. What is happening mathematically?** | Logistic regression maps a linear score through the sigmoid function to a value in (0,1), interpretable as a model-estimated probability. |
| **6. What happens if we change it?** | Changing the threshold trades precision against recall -- demonstrated directly below with our data's actual class balance. |


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score, roc_curve

FEATURES = [
    # NOTE: "age" is deliberately excluded here -- see the governance note above.
    "annual_income", "monthly_debt", "loan_amount", "loan_term_months",
    "credit_score", "employment_years", "account_age_months",
    "num_previous_loans", "previous_defaults", "debt_to_income", "loan_to_income",
]

X = df[FEATURES]
y = df["default"]

# THIS split (same SEED, same stratify, same test_size) is the split Module 2 reuses via the saved artifact.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=SEED
)
print("Train size:", len(X_train), " Test size:", len(X_test))

baseline_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=2000)),
])
baseline_pipeline.fit(X_train, y_train)

test_preds = baseline_pipeline.predict(X_test)
test_probs = baseline_pipeline.predict_proba(X_test)[:, 1]

print(confusion_matrix(y_test, test_preds))
print()
print(classification_report(y_test, test_preds, target_names=["No default", "Default"]))
print("ROC-AUC:", round(roc_auc_score(y_test, test_probs), 4))


**Honest read of ROC-AUC ≈ 0.70:** this is a real, usable-but-modest baseline -- meaningfully better than a coin flip (0.50), clearly worse than the strong, almost-too-clean scores (0.85-0.95) you sometimes see on synthetic teaching datasets. This is what *real* financial data with moderate signal actually looks like, and setting that expectation now is exactly why we validated this dataset from scratch instead of assuming a textbook-clean result. **Module 2's ANN will be measured against this exact number** -- do not expect it to dramatically exceed 0.70 either; if it does not, that is a legitimate, useful finding (see Module 2's own discussion of "when a neural network doesn't automatically win").

### Fix carried over: actually tuning the decision threshold


In [ ]:
from sklearn.metrics import precision_recall_curve

precisions, recalls, thresholds = precision_recall_curve(y_test, test_probs)

plt.figure(figsize=(7, 4))
plt.plot(thresholds, precisions[:-1], label="Precision", color="tab:blue")
plt.plot(thresholds, recalls[:-1], label="Recall", color="tab:red")
plt.axvline(0.5, color="grey", linestyle="--", alpha=.6, label="Default threshold (0.5)")
plt.xlabel("Decision threshold"); plt.ylabel("Score")
plt.title("Precision/Recall trade-off as the decision threshold changes")
plt.legend(); plt.grid(alpha=.3); plt.tight_layout(); plt.show()

fpr, tpr, roc_thresholds = roc_curve(y_test, test_probs)
youden_j = tpr - fpr
best_idx = np.argmax(youden_j)
best_threshold = roc_thresholds[best_idx]
print(f"Example data-driven threshold (max TPR-FPR): {best_threshold:.3f}  (vs. the naive default of 0.5)")
print("Notice this threshold sits BELOW 0.5 -- with a 35% base default rate (not a rare event), the")
print("statistically 'balanced' operating point naturally shifts down from the textbook 0.5 default.")
print("A real deployment must still weigh this against the ACTUAL business cost of a missed default")
print("vs. a false alarm -- a statistical threshold is a starting point, not the final answer.")


---
## Lesson 8 — Imbalanced classification + SMOTE: why accuracy can lie

| Question | Answer |
|---|---|
| **1. What problem are we solving?** | Even a *moderate* imbalance (our 35% default rate) can make naive accuracy misleading -- less dramatically than a rare-event problem, but still worth checking. |
| **2. Why does it matter in finance?** | Fraud, severe default, suspicious activity are often *far* rarer than 35% in real portfolios -- this lesson's techniques matter more as imbalance gets more severe, so it is worth learning here even though our own data is only moderately imbalanced. |
| **3. Why this technique?** | SMOTE creates synthetic minority-class training samples; `class_weight="balanced"` is a lower-cost alternative we compare directly. |
| **4. What do the numbers mean?** | SMOTE's neighbour settings control how synthetic samples are constructed -- there is no universal "correct" oversampling ratio. |
| **5. What is happening mathematically?** | SMOTE creates synthetic points *between* minority-class observations in feature space, rather than duplicating rows. |
| **6. What happens if we change it?** | Changing the oversampling strategy changes only the *training* distribution -- we always evaluate on an untouched, naturally distributed test set. |


In [ ]:
default_rate = y_train.mean()
print(f"Training default rate: {default_rate:.2%}")
naive_all_zero_accuracy = 1 - default_rate
print(f"A model predicting 'no default' for EVERYONE would score accuracy: {naive_all_zero_accuracy:.2%}")
print("...while catching ZERO actual defaults. Less dramatic than a 95%+ rare-event trap, but the")
print("same underlying lesson: accuracy alone is not a reliable metric once classes are unequal.")


In [ ]:
from imblearn.over_sampling import SMOTE
from collections import Counter

print("Before SMOTE:", Counter(y_train))
smote = SMOTE(random_state=SEED)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)
print("After SMOTE: ", Counter(y_train_smote))

scaler_for_smote = StandardScaler().fit(X_train)
smote_model = LogisticRegression(max_iter=2000)
smote_model.fit(scaler_for_smote.transform(X_train_smote), y_train_smote)
smote_probs = smote_model.predict_proba(scaler_for_smote.transform(X_test))[:, 1]
print("\nSMOTE-trained model -- Test ROC-AUC:", round(roc_auc_score(y_test, smote_probs), 4))


In [ ]:
class_weight_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=2000, class_weight="balanced")),
])
class_weight_pipeline.fit(X_train, y_train)
cw_probs = class_weight_pipeline.predict_proba(X_test)[:, 1]

print("Baseline (no imbalance handling) -- Test ROC-AUC:", round(roc_auc_score(y_test, test_probs), 4))
print("SMOTE-resampled training data    -- Test ROC-AUC:", round(roc_auc_score(y_test, smote_probs), 4))
print("class_weight='balanced'          -- Test ROC-AUC:", round(roc_auc_score(y_test, cw_probs), 4))
print()
print("On THIS dataset -- moderately imbalanced, only 800 rows -- neither technique dramatically")
print("outperforms the plain baseline. That is a genuinely useful, honest finding: imbalance-handling")
print("earns its complexity most clearly on SEVERE imbalance (think 2-5% fraud rates), not a 35% one.")
print("class_weight='balanced' still costs one keyword argument versus SMOTE's extra synthetic-data")
print("step -- always try it first.")


---
## Lesson 9 — K-Means++: discovering customer groups

| Question | Answer |
|---|---|
| **1. What problem are we solving?** | Discover groups of similar customers when there is **no target label** to predict. |
| **2. Why does it matter in finance?** | A financial institution may segment customers by behaviour or portfolio characteristics without a pre-existing segment label. |
| **3. Why this technique?** | K-Means++ improves initialisation over plain K-Means by choosing better-spread starting centroids. |
| **4. What do the parameters mean?** | `n_clusters` is a **modelling choice** -- validated below with silhouette score across a range, not assumed. |
| **5. What is happening mathematically?** | Alternates: assign each point to its nearest centroid, then move each centroid to the mean of its assigned points. |
| **6. What happens if we change it?** | Too few clusters merges meaningful populations; too many creates artificial fragments. |


In [ ]:
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import silhouette_score

cluster_features = df[["annual_income", "loan_amount", "credit_score", "debt_to_income"]]
scaled_cluster_features = StandardScaler().fit_transform(cluster_features)

silhouette_by_k = {}
for k in range(2, 8):
    km_trial = KMeans(n_clusters=k, init="k-means++", n_init=10, random_state=SEED)
    labels_trial = km_trial.fit_predict(scaled_cluster_features)
    silhouette_by_k[k] = silhouette_score(scaled_cluster_features, labels_trial)

for k, score in silhouette_by_k.items():
    print(f"K={k}: silhouette score = {score:.4f}")

best_k = max(silhouette_by_k, key=silhouette_by_k.get)
print(f"\nBest K by silhouette score: {best_k}")

plt.figure(figsize=(6, 3.5))
plt.plot(list(silhouette_by_k.keys()), list(silhouette_by_k.values()), marker="o")
plt.xlabel("K (number of clusters)"); plt.ylabel("Silhouette score")
plt.title("Validating K instead of assuming it"); plt.grid(alpha=.3)
plt.show()


In [ ]:
kmeans = KMeans(n_clusters=best_k, init="k-means++", n_init=10, random_state=SEED)
clusters = kmeans.fit_predict(scaled_cluster_features)
df["customer_cluster"] = clusters

print(df[["customer_id", "customer_cluster"]].head(10))
print("\nCluster sizes:")
print(df["customer_cluster"].value_counts().sort_index())


> **Honest read:** with only 800 rows across 4 features, the silhouette scores here (roughly 0.2-0.24) indicate *modest* cluster separation -- real customer segments rarely form perfectly distinct blobs, and a small dataset makes this harder still. The validated `K` is still the right K *for this data*, even if the resulting segments would benefit from more rows and features before being used for an actual business decision.


---
## Lesson 10 — DBSCAN: why you cannot blindly copy demo parameters

| Question | Answer |
|---|---|
| **1. What problem are we solving?** | Identify dense regions of similar customers, and explicitly mark points that do not belong to any dense region. |
| **2. Why does it matter in finance?** | Unlike K-Means, DBSCAN does not force every point into a cluster -- it can tell you "this customer doesn't fit any pattern we've seen," which is itself useful information. |
| **3. Why this technique?** | DBSCAN does not require specifying the number of clusters in advance. |
| **4. What do the parameters mean?** | `eps` is a neighbourhood radius (under the chosen scaling); `min_samples` is the density requirement. **Both are highly data-dependent -- watch what happens below when we copy settings from a different dataset without re-validating them.** |
| **5. What is happening mathematically?** | A point is part of a dense region when enough neighbours fall within its `eps` neighbourhood; sparse points become noise. |
| **6. What happens if we change it?** | This is the entire point of this lesson -- see the demonstration below. |


In [ ]:
# Step 1: naively copy eps/min_samples that "looked reasonable" without validating against THIS data
dbscan_naive = DBSCAN(eps=0.7, min_samples=10).fit(scaled_cluster_features)
naive_noise = (dbscan_naive.labels_ == -1).sum()
print(f"Naive copy-pasted parameters (eps=0.7, min_samples=10): {naive_noise} of {len(df)} points "
      f"({naive_noise/len(df):.1%}) labelled as noise.")
print("That is nearly half the dataset -- a strong signal these parameters do not fit THIS data's scale and density.")


In [ ]:
# Step 2: validate properly -- sweep candidate settings and require BOTH a reasonable noise fraction
# and at least 2 real clusters, exactly the discipline we applied to K in the previous lesson.
candidates = []
for eps in [0.6, 0.8, 1.0, 1.2, 1.5, 1.8, 2.0]:
    for min_samples in [5, 10, 15]:
        trial = DBSCAN(eps=eps, min_samples=min_samples).fit(scaled_cluster_features)
        noise_frac = (trial.labels_ == -1).sum() / len(df)
        n_clusters = len(set(trial.labels_)) - (1 if -1 in trial.labels_ else 0)
        if noise_frac < 0.15 and n_clusters >= 2:
            candidates.append((eps, min_samples, noise_frac, n_clusters))

for eps, ms, noise_frac, n_clusters in candidates:
    print(f"eps={eps}, min_samples={ms}: noise={noise_frac:.1%}, clusters={n_clusters}")

best_eps, best_min_samples = candidates[0][0], candidates[0][1]
print(f"\nSelected: eps={best_eps}, min_samples={best_min_samples}")

dbscan = DBSCAN(eps=best_eps, min_samples=best_min_samples).fit(scaled_cluster_features)
print("Labels found:", sorted(set(dbscan.labels_)))
print("Noise points:", (dbscan.labels_ == -1).sum(), f"of {len(df)} ({(dbscan.labels_==-1).sum()/len(df):.1%})")


**This is the lesson, made concrete instead of just stated:** the exact same algorithm, on the exact same data, went from "flagging half the dataset as noise" to "a sensible ~4% noise rate" purely by validating `eps`/`min_samples` against *this* dataset instead of reusing numbers that worked somewhere else. Every distance-based algorithm in this course (DBSCAN here, K-Means's `n_clusters`, Isolation Forest's `contamination` next) needs this same discipline.


---
## Lesson 11 — PCA: reducing many dimensions to a view we can inspect

| Question | Answer |
|---|---|
| **1. What problem are we solving?** | Reduce correlated numeric dimensions into a smaller representation for visualisation. |
| **2. Why does it matter in finance?** | Financial datasets can contain many correlated measures; analysts may need a 2D view before building more complex systems. |
| **3. Why this technique?** | PCA is fast, deterministic, and mathematically interpretable as a linear projection capturing directions of maximum variance. |
| **4. What do the parameters mean?** | `n_components=2` is chosen so humans can plot two dimensions -- not a claim the data truly has only two dimensions. |
| **5. What is happening mathematically?** | PCA finds orthogonal directions ordered by explained variance; the transformed coordinates are projections onto those directions. |
| **6. What happens if we change it?** | More components retain more information but reduce compression; fewer components ease visualisation while discarding more variance. |


In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
X_pca = pca.fit_transform(scaled_cluster_features)

print("Explained variance ratio:", pca.explained_variance_ratio_.round(4))
print("Total variance captured by 2 components:", pca.explained_variance_ratio_.sum().round(4))

plt.figure(figsize=(6, 5))
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=df["default"], cmap="coolwarm", alpha=0.5, s=15)
plt.xlabel("PC1"); plt.ylabel("PC2")
plt.title("PCA projection, coloured by actual default outcome")
plt.colorbar(scatter, label="default")
plt.show()


---
## Lesson 12 — UMAP: nonlinear structure for exploration

| Question | Answer |
|---|---|
| **1. What problem are we solving?** | Create a low-dimensional representation that can expose *nonlinear* structure PCA's linear projection would miss. |
| **2. Why does it matter in finance?** | Complex financial behaviour may contain nonlinear relationships hard to see with raw features or a linear projection alone. |
| **3. Why this technique?** | UMAP preserves local structure while producing a compact embedding -- good for exploration, but never a substitute for statistical validation. |
| **4. What do the parameters mean?** | `n_components=2` creates a 2D embedding; `random_state` fixes stochastic behaviour for reproducibility. |
| **5. What is happening mathematically?** | UMAP builds a graph-like representation of local relationships and optimises a low-dimensional embedding that tries to preserve them. |
| **6. What happens if we change it?** | Different neighbourhood/embedding settings can reveal substantially different structures -- results need cautious interpretation. |

**Install note:** `umap-learn` depends on `numba`, which can be sensitive to your installed NumPy version. If installation fails, try re-running the setup cell at the top of this notebook in a clean virtual environment.


In [ ]:
import umap

reducer = umap.UMAP(n_components=2, random_state=SEED)
embedding = reducer.fit_transform(scaled_cluster_features)

plt.figure(figsize=(6, 5))
scatter = plt.scatter(embedding[:, 0], embedding[:, 1], c=df["default"], cmap="coolwarm", alpha=0.5, s=15)
plt.title("UMAP embedding, coloured by actual default outcome")
plt.colorbar(scatter, label="default")
plt.show()


> **Interpretation warning:** a visually interesting UMAP plot is not proof that a real business segment exists -- treat it as exploratory evidence, the same discipline Lesson 9/10 applied to K-Means and DBSCAN.


---
## Lesson 13 — Data poisoning: attacking the learning process

Now we attack the foundation we spent Lessons 1-4 validating: the training data itself.

| Question | Answer |
|---|---|
| **1. What problem are we solving?** | Understand how manipulated training examples cause a model to learn incorrect patterns. |
| **2. Why does it matter in finance?** | If training data is corrupted, an enterprise can deploy a model that appears technically valid but encodes malicious or incorrect behaviour -- the exact opposite of what Lessons 1-4 tried to guarantee. |
| **3. Why this technique?** | Label flipping is easy to understand and reproduce -- a deliberately simple starting point. |
| **4. What do the parameters mean?** | A poisoning rate such as 5% is an experimental attack intensity, not a claim about how real attacks behave. |
| **5. What is happening mathematically?** | Label flipping changes the target `y` while leaving the input `X` unchanged. |
| **6. What happens if we change it?** | We sweep multiple rates so the degradation trend is visible -- on a small, 800-row dataset, expect this trend to be noisier than on a larger one, which is itself an honest finding about how attack visibility scales with data size. |


In [ ]:
clean_model = LogisticRegression(max_iter=2000)
clean_model.fit(baseline_pipeline.named_steps["scaler"].transform(X_train), y_train)
clean_prob = clean_model.predict_proba(baseline_pipeline.named_steps["scaler"].transform(X_test))[:, 1]
clean_auc = roc_auc_score(y_test, clean_prob)
print("Clean ROC-AUC:", round(clean_auc, 4))


In [ ]:
poison_results = {}
X_train_scaled = baseline_pipeline.named_steps["scaler"].transform(X_train)
X_test_scaled = baseline_pipeline.named_steps["scaler"].transform(X_test)

for poison_rate in [0.05, 0.10, 0.15, 0.20, 0.30]:
    poison_count = int(len(y_train) * poison_rate)
    poison_indices = rng.choice(len(y_train), size=poison_count, replace=False)

    y_train_poisoned = y_train.to_numpy().copy()
    y_train_poisoned[poison_indices] = 1 - y_train_poisoned[poison_indices]

    poisoned_model = LogisticRegression(max_iter=2000)
    poisoned_model.fit(X_train_scaled, y_train_poisoned)
    poison_prob = poisoned_model.predict_proba(X_test_scaled)[:, 1]
    poison_auc = roc_auc_score(y_test, poison_prob)
    poison_results[poison_rate] = poison_auc
    print(f"Poison rate {poison_rate:.0%}: AUC = {poison_auc:.4f}  (change vs clean: {poison_auc - clean_auc:+.4f})")

plt.figure(figsize=(6, 3.5))
plt.plot([clean_auc] + list(poison_results.values()), marker="o")
plt.xticks(range(len([0]+list(poison_results.keys()))), ["0% (clean)"] + [f"{r:.0%}" for r in poison_results.keys()])
plt.xlabel("Fraction of training labels flipped"); plt.ylabel("Test ROC-AUC")
plt.title("Model quality under increasing label-poisoning rate")
plt.grid(alpha=.3); plt.show()

print("\nOn a small, moderately-signalled real dataset like ours, the degradation trend is noisier than")
print("a large synthetic dataset would show -- but the overall direction still holds, and at 30% poisoning")
print("the damage is clear. Small real datasets are, if anything, MORE vulnerable to poisoning in practice,")
print("because each individual poisoned row carries more relative weight.")


> **Security lesson:** the attacker never needed access to the model API. They changed the information used to *teach* the model. This is why dataset access control, provenance, validation and versioning are security controls -- and it is the reason Lesson 15 builds a layered defence rather than relying on any single check.


---
## Lesson 14 — Isolation Forest: screening for unusual records

| Question | Answer |
|---|---|
| **1. What problem are we solving?** | Find potentially unusual observations when we do not have reliable labels for every observation -- including possible poisoning or data-entry errors, without knowing in advance which rows are affected. |
| **2. Why does it matter in finance?** | A financial organisation may have far more records than it can manually review; anomaly detection helps prioritise. |
| **3. Why this technique?** | Isolation Forest is purpose-built around isolating unusual observations, and does not require a fraud label for every training record. |
| **4. What do the parameters mean?** | `contamination=0.05` is an illustrative assumption for thresholding, not a universal constant. |
| **5. What is happening mathematically?** | Randomly partitioning feature space tends to isolate unusual observations with fewer splits. |
| **6. What happens if we change it?** | Changing `contamination` changes the outlier threshold -- validate it the same way we validated K and eps above, rather than trusting the default blindly. |


In [ ]:
from sklearn.ensemble import IsolationForest

anomaly_features = df[[
    "annual_income", "monthly_debt", "loan_amount",
    "credit_score", "debt_to_income", "loan_to_income",
]]

detector = IsolationForest(contamination=0.05, random_state=SEED)
flags = detector.fit_predict(anomaly_features)
df["anomaly_flag"] = flags

print(df["anomaly_flag"].value_counts())
print()
print(df[df["anomaly_flag"] == -1][
    ["customer_id", "annual_income", "loan_amount", "credit_score", "debt_to_income",
     "flag_employment_age_inconsistent"]
].head(10))


In [ ]:
# Does the Isolation Forest's anomaly flag overlap with our own Lesson 2 consistency flag?
overlap = df[(df["anomaly_flag"] == -1) & (df["flag_employment_age_inconsistent"])]
print(f"Records flagged by BOTH the employment/age check AND Isolation Forest: {len(overlap)}")
print("A meaningful overlap here would strengthen confidence in both signals; a small overlap tells us")
print("the two checks are catching DIFFERENT kinds of unusual records -- both are worth keeping.")


> **Never equate anomaly with fraud.** An unusually large loan-to-income ratio may be a legitimate large purchase. Anomaly detection is a **triage signal** -- fraud/quality determination needs additional evidence and human review.


---
## Lesson 15 — From individual checks to a layered defence

| Question | Answer |
|---|---|
| **1. What problem are we solving?** | Turn Lessons 1-14's individual checks into one coherent architecture, instead of a pile of disconnected scripts. |
| **2. Why does it matter in finance?** | An organisation needs reproducibility and evidence: who changed the dataset, which version trained the model, what validation occurred, what was quarantined. |
| **3. Why this technique?** | A **layered** defence beats relying on any single check -- exactly what this entire notebook has been building, lesson by lesson. |
| **4. What do the parameters mean?** | Thresholds should be selected based on validation and operational capacity, as demonstrated throughout this notebook (K, eps, contamination, decision threshold). |
| **5. What is happening mathematically?** | The pipeline separates collection -> validation -> anomaly screening -> quarantine -> approval -> training. |
| **6. What happens if we change it?** | Removing one layer changes the residual risk -- no single check in this notebook would have caught everything alone. |

```
Raw dataset → Schema & quality checks (L1) → Logical consistency (L2) → Business-logic label check (L4)
           → Anomaly screening (L14) → Quarantine/review → Training
```

Look back at what each lesson actually caught:

| Lesson | Check | What it would catch that others miss |
|---|---|---|
| 1 | Missing values, duplicates | Structural gaps |
| 2 | Cross-field consistency | Logically impossible combinations (age vs. employment) |
| 4 | Label-vs-risk-factor direction | A corrupted or mislabelled target column |
| 13 | Poisoning simulation | Deliberate label manipulation, at any rate |
| 14 | Isolation Forest | Statistically unusual combinations, without needing a rule for every case |

No single layer is sufficient alone -- that is the point.


In [ ]:
review_queue = df[(df["anomaly_flag"] == -1) | (df["flag_employment_age_inconsistent"])].copy()
print(f"{len(review_queue)} of {len(df)} records ({len(review_queue)/len(df):.1%}) routed to human review "
      f"(anomaly flag OR employment/age inconsistency).")
review_queue[[
    "customer_id", "annual_income", "loan_amount", "credit_score", "debt_to_income",
    "anomaly_flag", "flag_employment_age_inconsistent",
]].head(15)


---
## Hand-off: saving everything Module 2 (and every later module) will build on


In [ ]:
enriched_path = Path("data/rr_finance_module1_dataset_enriched.csv")
df.to_csv(enriched_path, index=False)
print("Saved:", enriched_path.resolve())

baseline_path = Path("artifacts/baseline_logreg_pipeline.joblib")
joblib.dump(baseline_pipeline, baseline_path)
print("Saved:", baseline_path.resolve())

iso_path = Path("artifacts/isolation_forest.joblib")
joblib.dump(detector, iso_path)
print("Saved:", iso_path.resolve())

metrics_summary = {
    "module": 1,
    "dataset_rows": int(len(df)),
    "default_rate": float(df["default"].mean()),
    "rows_flagged_employment_age": int(df["flag_employment_age_inconsistent"].sum()),
    "baseline_logreg_test_auc": float(roc_auc_score(y_test, test_probs)),
    "smote_logreg_test_auc": float(roc_auc_score(y_test, smote_probs)),
    "class_weight_logreg_test_auc": float(roc_auc_score(y_test, cw_probs)),
    "kmeans_best_k": int(best_k),
    "kmeans_best_silhouette": float(silhouette_by_k[best_k]),
    "dbscan_eps": float(best_eps),
    "dbscan_min_samples": int(best_min_samples),
    "isolation_forest_contamination": 0.05,
    "isolation_forest_flagged_records": int((df["anomaly_flag"] == -1).sum()),
    "poisoning_auc_by_rate": {str(k): float(v) for k, v in poison_results.items()},
    "features_used": FEATURES,
    "random_seed": SEED,
    "known_limitations": [
        "Only 800 rows -- treat metrics as directional, not production-grade.",
        "4.25% of rows flagged for an employment-years/age inconsistency; kept but tagged, not removed.",
        "Baseline ROC-AUC ~0.70 reflects genuinely modest feature-target signal, not a modelling error.",
    ],
}
metrics_path = Path("artifacts/module1_metrics.json")
with open(metrics_path, "w") as f:
    json.dump(metrics_summary, f, indent=2)
print("Saved:", metrics_path.resolve())

print("\n--- Module 1 hand-off complete ---")
for k, v in metrics_summary.items():
    print(f"  {k}: {v}")


In [ ]:
reloaded_pipeline = joblib.load("artifacts/baseline_logreg_pipeline.joblib")
reloaded_auc = roc_auc_score(y_test, reloaded_pipeline.predict_proba(X_test)[:, 1])
print("Original baseline AUC: ", round(roc_auc_score(y_test, test_probs), 6))
print("Reloaded pipeline AUC: ", round(reloaded_auc, 6))
assert abs(reloaded_auc - roc_auc_score(y_test, test_probs)) < 1e-9
print("Match confirmed -- this artifact is safe for Module 2 to load and trust.")


## Module 1 hand-off summary

| Artifact | Location | What it is |
|---|---|---|
| Validated dataset | `data/rr_finance_module1_dataset.csv` | The original CSV, with two quality flag columns added |
| Enriched dataset | `data/rr_finance_module1_dataset_enriched.csv` | Same table + `customer_cluster` + `anomaly_flag` |
| Baseline model | `artifacts/baseline_logreg_pipeline.joblib` | Fitted `StandardScaler` + `LogisticRegression`, ROC-AUC ≈ 0.70 |
| Anomaly detector | `artifacts/isolation_forest.joblib` | Fitted `IsolationForest`, validated `contamination` |
| Metrics summary | `artifacts/module1_metrics.json` | Every headline number from this notebook, plus documented limitations |

### On dataset size -- a note for growing this system beyond the classroom

800 rows was enough to teach every technique in this module honestly, and to demonstrate real, non-fabricated data-quality findings (the employment/age inconsistency, the DBSCAN parameter lesson). It is **not** enough for a production credit model. If RR Finance moves this system toward real deployment, the highest-value next step is not a fancier algorithm -- it is **more validated rows of the same shape**, collected and quality-checked the same way this notebook checked these 800. A larger public dataset (e.g. Kaggle's "Give Me Some Credit," ~150,000 rows, or the UCI German Credit dataset) could supplement this data for practicing techniques at scale, but would need its own Lesson 1-4 validation pass before being trusted -- the *process* in this notebook, not just this specific CSV, is the reusable asset.

### What Module 2 does with this

Module 2 (Deep Neural Networks) opens by **loading** `data/rr_finance_module1_dataset.csv` and `artifacts/baseline_logreg_pipeline.joblib` -- not regenerating them. It trains a multi-layer ANN on the *same* train/test split this notebook used, and benchmarks it against the *exact* baseline AUC (~0.70) saved above.

**Before moving to Module 2:** confirm `data/` and `artifacts/` in your project folder contain the five files listed in the table above.
